# Ellipsoid Localizer
This notebook explains the different hyperparameters of the Ellipsoid Localizers and does ablaition studies on some of them. For a rough idea on how to use localizers in general refer to the Quickstart.ipynb

### Hyperparameter Overview:

`extract_and_match_wrapper_config`: How to get the initial PnP based guess, standard is LightGlue

`cam1_segmenter`, `cam2_segmenter`: Either an SAM3Segmenter or YOLOv26Segmenter

- Will default to standard Sam3 + YOLOv26, using 2 scene adjusted YOLOv26 instances is usually a good idea.

- Sam3 takes a SAM3Prompt(text = ...), the text describes the single concept prompting for, standard is "medium sized object"

- YOLOv26, takes the `model` type and `prompts`, a list of concepts, it is highly recommended to tune it to the scene.


`pne_optimizer`:

- How to optimize the pose based on ellipse-ellipsoid wasserstein distance

- See the section `Influence of different optimizers` for the different ones

- Leaving it as PyposePNEOptimizer() is the most stable and effective version.


`min_number_matched_ellipsoids_for_opt`: How many matched ellipsoids are needed for optimization, higher -> less often refined. Values >= 3 remove most bad optimizations.

`ellipsoid_refinement_at_res`: The resolution on which the segmenters will be run. Leaving at None is good enough for most applications.

`matching_config`: How to match the projected ellipsoids to the observed ellipses

- `dummy_value`: Lower -> less false positives and less true positives. Values <0.01 are normal. If you visualize matching and there are some weird matches in some frames / permutations adjust this. 

- `k`: Low values (~1.5) will remove matches with unusually high costs, mostly not needed (leave at 1000)

`ellipsoid_matching_config`: How to merge the segmented 3D point clouds into point clouds representing the 3D objects

- `max_centroid_dist`, maximum distance between centroids of point_clouds to merge them, increase if there are multiple ellipsoids for each object

- `max_color_dist`, maximum color distance between point clouds, to be merged.

- `min_cluster_size`, min number of scan viewpoints to see an object for it to be used. Highly recommended to set to at least 2, to remove ghost objects

- `min_number_points_per_detected_object`, min number of points to form an ellipsoid. Increase to remove small objects / partial scans.

`ellipsoid_fitter`: How to fit the ellipsoids to the point clouds
- `SimpleEllipsoidFitter`: the `contamination` in [0,0.5] removes outliers in each object (bigger -> more removed). `min_number_points` is the number of points an object after clean up has to have, to generate an ellipsoid.

- `MVEEEllipsoidFitter`: `tolerance` is the convergence threshhold (bigger -> faster), `max_iterations` is the number of iterations and `use_convex_hull` wheather to use the hull to speed things up (highly recommended). The `contamination` can pretty effectively indirectly tune the size of the fitted ellipsoids, this can be useful when adjusting the signed z-axis error.

- `LeastShellDistanceEllipsoidFitter`: For defining the error function: `size_penalty` stops sizes from exploding, <0.1 is common, `distance_p_norm` and `size_p_norm`, define the aggregation of the errors. Choose the 1 norm for best outlier resistence. For pose optimization leaving it to `se3_exp` is advised (in experience doesnt fit a giant ellipsoid to shell due to early stagnation). `adam_config` defines the SGD parameters.

`visualize_pne_optimisation`, `visualize_matching`, `visualize_environment_generation`, `visualize_segmentation_masks`: Wheather to visualize different steps (for debugging), visualizing the environment generation is expecially helpful to optimize fitting hyperparameters.

### Notebook initialisation

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import seaborn as sns
sns.set_theme(style="whitegrid")


import numpy as np
import random, torch, os, cv2

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import matplotlib.pyplot as plt
from headset_localization import *

### Loading the Dataset

In [ ]:
from headset_localization import *
from shared.complete_robot_scan import CompleteRobotScan
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

#robot_data_folder_location = "../example_datasets/small_aruco_2"
#vrs_file_location = "../example_datasets/small_aruco_2_2.vrs"


# Load robot scan and generate a 3D reconstructed environment
robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=True)
)

# Label the headset data
labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

# If you want to visualize the loaded data
visualize_loaded_data = False
if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)


## Testing Localizer variations
Running a single localizer on the loaded dataset:

In [ ]:
predictor = EllipsoidLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=ExtractAndLightGlue(),
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PnEDeltaPoseAdamOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=YOLOv26Segmenter("yoloe-26l-seg.pt"),
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_segmentation_masks=False,
        visualize_pne_optimisation=False,
        visualize_environment_generation=True
)

In [ ]:
init_predictor_grade = PredictionOnDataset(
    predictor = predictor,
    headset_data = labeled_headset_data,
    number_retry = 1
)
init_predictor_grade.print_summary()


visualize_prediction = True
if visualize_prediction:
    init_predictor_grade.visualize_predictions(
        robot_env=robot_env,
        show_label=False
    )

### Influence of different optimizers
This tests different pose optimization backends:

In [ ]:
from headset_localization import *

light_glue = ExtractAndLightGlue()
yolo = YOLOv26Segmenter("yoloe-26l-seg.pt")

optimizers = [
    PyposePNEOptimizer(accumulate_losses=True, config = PyposePnEOptimizerConfig(device = "cpu", convergence_threshold=1e-6, lm_max_steps=25)),
    PnEDeltaPoseAdamOptimizer(accumulate_losses = True, device='cpu', delta_pose_mapping = 'rpy', adam_cfg=AdamConfig(
        learning_rate=0.001, convergence_threshold=0.00000001, max_itterations=90
    )),
    PnEDeltaPoseAdamOptimizer(accumulate_losses = True, device='cpu', delta_pose_mapping='se3_exp', adam_cfg=AdamConfig(
        learning_rate=0.005, convergence_threshold=0.00000001
    )),
    PnEDeltaPoseAdamOptimizer(accumulate_losses = True, device='cpu', delta_pose_mapping = 'rpy', opt_datatype = torch.float64, adam_cfg=AdamConfig(
        learning_rate=0.001, convergence_threshold=0.00000001, max_itterations=90
    )),
    PnEDeltaPoseAdamOptimizer(accumulate_losses = True, device='cpu', delta_pose_mapping='se3_exp', opt_datatype = torch.float64, adam_cfg=AdamConfig(
        learning_rate=0.005, convergence_threshold=0.00000001, max_itterations=90
    )),
    PnEDeltaPoseLBFGSOptimizer(accumulate_losses = True, delta_pose_mapping='rpy', opt_datatype=torch.float64, config=PnEDeltaPoseLBFGSOptimizerConfig(
        learning_rate=0.01,
        max_itterations = 80,
        history_size=7,
        stop_at_grad=1e-12,
        convergence_threshold=1e-9,
        line_search_function = 'strong_wolfe'
    )),
    PnEDeltaPoseLBFGSOptimizer(accumulate_losses = True, delta_pose_mapping='se3_exp', opt_datatype=torch.float64, config=PnEDeltaPoseLBFGSOptimizerConfig(
        learning_rate=0.01,
        max_itterations = 80,
        history_size=7,
        stop_at_grad=1e-12,
        convergence_threshold=1e-9,
        line_search_function = 'strong_wolfe'
    )),
]

names = [
    "Pypose", "Adam - RPY - float32", "Adam - Lie - float32", "Adam - RPY - float64", "Adam - Lie - float64", "LBFGS - RPY - float64", "LBFGS - Lie - float64"
]

different_ellipse_optimizers = [
    GradableLocalizer(
        creator=EllipsoidLocalizer.get_creation_function(
            cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                rotation_augmentations=[Rotate180Deg],
                extract_and_match=light_glue,
                ransac_config=pose_estimation_ransaac_config_less_precise,
            ),
            pne_optimizer=optimizer,
            ellipsoid_refinement_at_res=(1400, 1400),
            cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
            cam2_segmenter=yolo,
            matching_config=GaussianMatchingConfig(dummy_value=0.01),
            ellipsoid_matching_config = PointCloudMatchingConfig(),
            ellipsoid_fitter=SimpleEllipsoidFitter(visualize=False, contamination=0.05),
            visualize_pne_optimisation=False,
        ),
        name=f"Ellipse: {name}"
    ) for name, optimizer in zip(names, optimizers)
] + [
    GradableLocalizer(
        creator=PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                rotation_augmentations=[Rotate180Deg],
                extract_and_match=light_glue,
                ransac_config=pose_estimation_ransaac_config_less_precise,
            )
        ),    name=f"PnP"
    )
]


diff_optim_grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_ellipse_optimizers,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
diff_optim_grader.print_summary()

#for optimizer in optimizers:
#    fig, ax = plt.subplots(1, 1, figsize = (12, 6))
#    optimizer.visualize_opt_losses(ax, log_scale=False)

fig3, ax3 = plt.subplots(1, 1, figsize = (6, 3))
diff_optim_grader.plot_prediction_times(ax3, rotate_x_labels=30)

fig3, ax3 = plt.subplots(1, 1, figsize = (6, 3))
diff_optim_grader.plot_prediction_times(ax3, rotate_x_labels=30, plot_legend=False)

fig4, ax4 = plt.subplots(1, 1, figsize = (12, 5.5))
visualize_multiple_pne_optimizer_losses(ax=ax4, optimizers=optimizers, use_log_scale=True, names=names)

### Testing different fitting methods
This section showcases the different ellipsoid fitters and their performance implications.

In [ ]:

pnp = GradableLocalizer(
    creator=PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_precise,
        )
    ),
    name="PnP"
)


ellipse_simple = GradableLocalizer(
    creator=EllipsoidLocalizer.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
            display_matching=False
        ),
        pne_optimizer=PyposePNEOptimizer(),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        ellipsoid_matching_config = PointCloudMatchingConfig(min_cluster_size=2),
        ellipsoid_fitter=SimpleEllipsoidFitter(),
        visualize_environment_generation = False
    ),
    name="Simple"
)


ellipse_shell_dist = GradableLocalizer(
    creator=EllipsoidLocalizer.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
            display_matching=False
        ),
        pne_optimizer=PyposePNEOptimizer(),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        ellipsoid_matching_config = PointCloudMatchingConfig(min_cluster_size=2),
        ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(visualize=False, contamination=0.05, delta_pose_mapping="se3_exp", size_penalty=0.02),
        visualize_environment_generation = True
    ),
    name="Least Shell Distance"
)

ellipse_mvee = GradableLocalizer(
    creator=EllipsoidLocalizer.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
            display_matching=False
        ),
        pne_optimizer=PyposePNEOptimizer(),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        ellipsoid_matching_config = PointCloudMatchingConfig(min_cluster_size=2),
        ellipsoid_fitter=MVEEEllipsoidFitter(visualize=False, contamination=0.05),
        visualize_environment_generation = False
    ),
    name="MVEE"
)

grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[pnp, ellipse_simple ,ellipse_shell_dist , ellipse_mvee],
    headset_data = labeled_headset_data,
    robot_env = robot_env,
    compute_ray_intersection_error=True
)

In [ ]:
grader.visualize_predictions_3d()

fig, axes = plt.subplots(1, 2, figsize = (12, 3.5))
grader.plot_creation_times(axes[0], plot_legend=True)
grader.plot_error_vs_error(axes[1], error_type_1=SingleValueErrorType.MED_TRANSLATIONAL, error_type_2=SingleValueErrorType.MED_ROTATIONAL, plot_legend=False, adjust_texts=False)


fig, ax = plt.subplots(1, 1, figsize = (12, 6))
grader.plot_time_series_error(ax, error_type=TimeSeriesErrorType.ABS_TRANSLATIONAL)

fig1, axes = plt.subplots(2, 3, figsize = (13, 5))
grader.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]],explain = True)

fig1, axes = plt.subplots(2, 3, figsize = (13, 5))
grader.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]],explain = False)

grader.print_summary()

### Using an ellipsoid localizer on the fr2/desk dataset

Loading the dataset: (first tenth)

In [ ]:
dataset_location_fr2_desk = "../tum_datasets/rgbd_dataset_freiburg2_desk"
from headset_localization import *


tum_robot_env, tum_headset_data = scanned_3d_environment_and_headset_recording_from_tum(
        folder=dataset_location_fr2_desk,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=False),
        intervall=(0.0, 0.1)
)

Initializing and grading different versions. 
Using a tuned YOLOv26 instance to do the job:

In [ ]:
yolo = YOLOv26Segmenter("yoloe-26l-seg.pt", prompts=["monitor", "keyboard", "mouse", "teddy", "tape", "book", "cup", "telephone", "can", "office appliance"])

pnp_tum = GradableLocalizer(
    creator=PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
            rotation_augmentations=[Augmentation],
            ransac_config=pose_estimation_ransaac_config_precise,
        )
    ),
    name="PnP"
)


ellipse_localizers_tum = [GradableLocalizer(
    creator=EllipsoidLocalizer.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
            rotation_augmentations=[Augmentation],
            ransac_config=pose_estimation_ransaac_config_less_precise,
            display_matching=False
        ),
        pne_optimizer=PyposePNEOptimizer(),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt(mask_threshold=0.8)),
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        ellipsoid_matching_config = PointCloudMatchingConfig(),
        ellipsoid_fitter=e_fitter,
        visualize_environment_generation = False,
        visualize_segmentation_masks=False,
        visualize_pne_optimisation=False
    ),
    name=name
) for e_fitter, name in zip(
        [LeastShellDistanceEllipsoidFitter(contamination=0.4, size_p_norm=2, visualize=False), SimpleEllipsoidFitter(contamination=0.4), MVEEEllipsoidFitter(contamination=0.4)],
        ["Least Shell Distance", "Simple", "MVEE"]
    )
]

tum_grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=ellipse_localizers_tum+[pnp_tum],
    headset_data = tum_headset_data,
    robot_env = tum_robot_env,
    compute_ray_intersection_error=True
)

In [ ]:
tum_grader.visualize_predictions_3d()

fig, axes = plt.subplots(1, 2, figsize = (12, 3.5))
tum_grader.plot_creation_times(axes[0], plot_legend=True)
tum_grader.plot_error_vs_error(axes[1], error_type_1=SingleValueErrorType.MED_TRANSLATIONAL, error_type_2=SingleValueErrorType.MED_ROTATIONAL, plot_legend=False, adjust_texts=False)


fig, ax = plt.subplots(1, 1, figsize = (12, 6))
tum_grader.plot_time_series_error(ax, error_type=TimeSeriesErrorType.ABS_TRANSLATIONAL)


fig, axes = plt.subplots(1, 2, figsize = (12, 3))
tum_grader.plot_error_vs_error(axes[0], error_type_1=SingleValueErrorType.MED_TRANSLATIONAL, error_type_2=SingleValueErrorType.MED_ROTATIONAL, plot_legend=False, adjust_texts=False)
tum_grader.plot_error_vs_error(axes[1], error_type_1=SingleValueErrorType.MED_TRANSLATIONAL, error_type_2=SingleValueErrorType.MED_ROTATIONAL, plot_legend=False, adjust_texts=False)

fig1, axes = plt.subplots(2, 3, figsize = (13, 5))
tum_grader.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]],explain = True)

fig1, axes = plt.subplots(2, 3, figsize = (13, 5))
tum_grader.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]],explain = False)

tum_grader.print_summary()